# EEGNet

In [1]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, AveragePooling2D, Flatten,
                                     Dense, BatchNormalization, Activation, Dropout, SeparableConv2D)
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
input_shape = (27, 2560, 1)
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
all_data_folder = "Original/All Data"

# --------------------------------------------------⚡️ Helper functions ---------------------------------------------------
def build_eegnet_model(input_shape, f1, d_mult, dropout_rate, lr, num_classes=1):
    """Builds the EEGNet model. The architecture is flexible to the new input shape."""
    inputs = Input(shape=input_shape)
    x = Conv2D(f1, (1, 64), padding='same', use_bias=False)(inputs)
    x = BatchNormalization()(x)
    # This DepthwiseConv2D layer is key; it performs the spatial convolution.
    # It correctly uses input_shape[0] (now 27) as the kernel height.
    x = DepthwiseConv2D((input_shape[0], 1), use_bias=False, depth_multiplier=d_mult, depthwise_constraint=max_norm(1.))(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropout_rate)(x)
    x = SeparableConv2D(f1 * d_mult, (1, 16), use_bias=False, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x)
    x = Dropout(dropout_rate)(x)
    x = Flatten()(x)
    x = Dense(num_classes, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=x)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

def load_data_from_folder(folder):
    """Loads file paths and labels from a specified folder as a Python list."""
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        """Use ceiling division to include the last, smaller batch."""
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        """Generates data containing batch_size samples."""
        data, labels = [], []
        for file, label in batch_data_list:
            sample = np.load(file) # Assumed shape: (2560, 27)
            
            # ✅ Transpose and reshape the data to match the new input_shape
            sample = sample.T # Transpose from (2560, 27) to (27, 2560)
            sample = np.expand_dims(sample, axis=-1) # Add channel dim to get (27, 2560, 1)
            
            data.append(sample)
            labels.append(label)

        # Stacking the list of correctly shaped samples creates the final batch
        data = np.array(data)
        labels = np.array(labels, dtype=np.float32)
        return data, labels

# --------------------------------------------------⚡️ Load all data ---------------------------------------------------
all_set = load_data_from_folder(all_data_folder)
print(f"📦 Total samples found: {len(all_set)}")
all_set = np.array(all_set, dtype=object)

# --------------------------------------------------⚡️ Objective Function for Optuna ---------------------------------------------------
def objective(trial):
    f1 = trial.suggest_categorical('f1_filters', [8, 16, 32])
    d_mult = trial.suggest_categorical('depth_multiplier', [2, 3, 4])
    dropout_rate = trial.suggest_float('dropout_rate', 0.25, 0.6)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    epochs = 70

    print(f"\n🚀 Starting Trial {trial.number} with params: F1={f1}, D_Mult={d_mult}, Dropout={dropout_rate:.3f}, LR={lr:.5f}, Batch={batch_size}")

    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_set)):
        train_data_list = all_set[train_idx]
        val_data_list = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data_list, batch_size=batch_size, shuffle=True)
        val_gen = EEGDataGenerator(val_data_list, batch_size=batch_size, shuffle=False)
        
        model = build_eegnet_model(input_shape, f1, d_mult, dropout_rate, lr)
        
        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=epochs,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # Get predictions from the generator
        y_pred_probs = model.predict(val_gen, verbose=0)
        
        # Extract true labels directly from the generator to guarantee alignment
        y_true = []
        for i in range(len(val_gen)):
            _, labels_batch = val_gen[i]
            y_true.extend(labels_batch)
        y_true = np.array(y_true)
        
        y_pred = (y_pred_probs > 0.5).astype(int)
        
        current_f1 = f1_score(y_true, y_pred)
        fold_f1_scores.append(current_f1)
        
    mean_f1_score = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1_score:.4f}")
    
    return mean_f1_score

# --------------------------------------------------⚡️ Run the Optuna Study ---------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# --------------------------------------------------⚡️ Print the results of the study ---------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")
print("\n🎉 Best trial:")
best_trial = study.best_trial
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}")
print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-08-10 17:38:07,778] A new study created in memory with name: no-name-435de74c-805c-4f5c-8718-1d9d238e0b24


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: F1=32, D_Mult=4, Dropout=0.367, LR=0.00010, Batch=32


[I 2025-08-10 20:11:37,409] Trial 0 finished with value: 0.8836702626039271 and parameters: {'f1_filters': 32, 'depth_multiplier': 4, 'dropout_rate': 0.36705475406061505, 'learning_rate': 0.00010217349320248059, 'batch_size': 32}. Best is trial 0 with value: 0.8836702626039271.


✅ Trial 0 finished with Mean F1-Score: 0.8837

🚀 Starting Trial 1 with params: F1=32, D_Mult=3, Dropout=0.453, LR=0.00029, Batch=16


[I 2025-08-10 22:05:55,669] Trial 1 finished with value: 0.8899648293939683 and parameters: {'f1_filters': 32, 'depth_multiplier': 3, 'dropout_rate': 0.4530017975297942, 'learning_rate': 0.0002937469880962401, 'batch_size': 16}. Best is trial 1 with value: 0.8899648293939683.


...

✅ Trial 7 finished with Mean F1-Score: 0.8858

🚀 Starting Trial 8 with params: F1=8, D_Mult=2, Dropout=0.526, LR=0.00541, Batch=64


[I 2025-08-11 08:01:30,140] Trial 8 finished with value: 0.8502754161193422 and parameters: {'f1_filters': 8, 'depth_multiplier': 2, 'dropout_rate': 0.5262932943501599, 'learning_rate': 0.005410725404145268, 'batch_size': 64}. Best is trial 3 with value: 0.891649739596879.


✅ Trial 8 finished with Mean F1-Score: 0.8503

🚀 Starting Trial 9 with params: F1=32, D_Mult=3, Dropout=0.257, LR=0.00021, Batch=32


[I 2025-08-11 09:39:53,921] Trial 9 finished with value: 0.8833127029975092 and parameters: {'f1_filters': 32, 'depth_multiplier': 3, 'dropout_rate': 0.25719086879804226, 'learning_rate': 0.0002102866664045226, 'batch_size': 32}. Best is trial 3 with value: 0.891649739596879.


✅ Trial 9 finished with Mean F1-Score: 0.8833

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.8916

  Best Hyperparameters:
    f1_filters: 16
    depth_multiplier: 3
    dropout_rate: 0.46178052725421637
    learning_rate: 0.0004449502500474054
    batch_size: 64


In [1]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, AveragePooling2D, Flatten,
                                     Dense, BatchNormalization, Activation, Dropout, SeparableConv2D)
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
input_shape = (27, 2560, 1)
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
all_data_folder = "Raw/All Data"

# --------------------------------------------------⚡️ Helper functions ---------------------------------------------------
def build_eegnet_model(input_shape, f1, d_mult, dropout_rate, lr, num_classes=1):
    """Builds the EEGNet model. The architecture is flexible to the new input shape."""
    inputs = Input(shape=input_shape)
    x = Conv2D(f1, (1, 64), padding='same', use_bias=False)(inputs)
    x = BatchNormalization()(x)
    # This DepthwiseConv2D layer is key; it performs the spatial convolution.
    # It correctly uses input_shape[0] (now 27) as the kernel height.
    x = DepthwiseConv2D((input_shape[0], 1), use_bias=False, depth_multiplier=d_mult, depthwise_constraint=max_norm(1.))(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropout_rate)(x)
    x = SeparableConv2D(f1 * d_mult, (1, 16), use_bias=False, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x)
    x = Dropout(dropout_rate)(x)
    x = Flatten()(x)
    x = Dense(num_classes, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=x)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

def load_data_from_folder(folder):
    """Loads file paths and labels from a specified folder as a Python list."""
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        """Use ceiling division to include the last, smaller batch."""
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        """Generates data containing batch_size samples."""
        data, labels = [], []
        for file, label in batch_data_list:
            sample = np.load(file) # Assumed shape: (2560, 27)
            
            # ✅ Transpose and reshape the data to match the new input_shape
            sample = sample.T # Transpose from (2560, 27) to (27, 2560)
            sample = np.expand_dims(sample, axis=-1) # Add channel dim to get (27, 2560, 1)
            
            data.append(sample)
            labels.append(label)

        # Stacking the list of correctly shaped samples creates the final batch
        data = np.array(data)
        labels = np.array(labels, dtype=np.float32)
        return data, labels

# --------------------------------------------------⚡️ Load all data ---------------------------------------------------
all_set = load_data_from_folder(all_data_folder)
print(f"📦 Total samples found: {len(all_set)}")
all_set = np.array(all_set, dtype=object)

# --------------------------------------------------⚡️ Objective Function for Optuna ---------------------------------------------------
def objective(trial):
    f1 = trial.suggest_categorical('f1_filters', [8, 16, 32])
    d_mult = trial.suggest_categorical('depth_multiplier', [2, 3, 4])
    dropout_rate = trial.suggest_float('dropout_rate', 0.25, 0.6)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    epochs = 70

    print(f"\n🚀 Starting Trial {trial.number} with params: F1={f1}, D_Mult={d_mult}, Dropout={dropout_rate:.3f}, LR={lr:.5f}, Batch={batch_size}")

    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_set)):
        print(f"--- Fold {fold + 1}/{n_splits} ---")
        train_data_list = all_set[train_idx]
        val_data_list = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data_list, batch_size=batch_size, shuffle=True)
        val_gen = EEGDataGenerator(val_data_list, batch_size=batch_size, shuffle=False)
        
        model = build_eegnet_model(input_shape, f1, d_mult, dropout_rate, lr)
        
        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=epochs,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # Get predictions from the generator
        y_pred_probs = model.predict(val_gen, verbose=0)
        
        # Extract true labels directly from the generator to guarantee alignment
        y_true = []
        for i in range(len(val_gen)):
            _, labels_batch = val_gen[i]
            y_true.extend(labels_batch)
        y_true = np.array(y_true)
        
        y_pred = (y_pred_probs > 0.5).astype(int)
        
        current_f1 = f1_score(y_true, y_pred)
        fold_f1_scores.append(current_f1)
        
    mean_f1_score = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1_score:.4f}")
    
    return mean_f1_score

# --------------------------------------------------⚡️ Run the Optuna Study ---------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# --------------------------------------------------⚡️ Print the results of the study ---------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")
print("\n🎉 Best trial:")
best_trial = study.best_trial
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}")
print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-08-11 17:01:22,116] A new study created in memory with name: no-name-6c975e14-aa9a-4b40-8566-70df4b715d19


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: F1=8, D_Mult=3, Dropout=0.461, LR=0.00623, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-08-11 17:58:38,428] Trial 0 finished with value: 0.8233195923192641 and parameters: {'f1_filters': 8, 'depth_multiplier': 3, 'dropout_rate': 0.46142700102493484, 'learning_rate': 0.006234375161569849, 'batch_size': 16}. Best is trial 0 with value: 0.8233195923192641.


✅ Trial 0 finished with Mean F1-Score: 0.8233

🚀 Starting Trial 1 with params: F1=8, D_Mult=3, Dropout=0.335, LR=0.00654, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-08-11 18:40:57,527] Trial 1 finished with value: 0.8271545132053465 and parameters: {'f1_filters': 8, 'depth_multiplier': 3, 'dropout_rate': 0.335125193359961, 'learning_rate': 0.006537777952849018, 'batch_size': 16}. Best is trial 1 with value: 0.8271545132053465.


...

✅ Trial 7 finished with Mean F1-Score: 0.8686

🚀 Starting Trial 8 with params: F1=32, D_Mult=4, Dropout=0.342, LR=0.00118, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-08-12 04:25:50,009] Trial 8 finished with value: 0.8911851558035646 and parameters: {'f1_filters': 32, 'depth_multiplier': 4, 'dropout_rate': 0.341693680712148, 'learning_rate': 0.0011756074074994365, 'batch_size': 32}. Best is trial 8 with value: 0.8911851558035646.


✅ Trial 8 finished with Mean F1-Score: 0.8912

🚀 Starting Trial 9 with params: F1=16, D_Mult=3, Dropout=0.442, LR=0.00183, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-08-12 06:01:17,318] Trial 9 finished with value: 0.8677912602660905 and parameters: {'f1_filters': 16, 'depth_multiplier': 3, 'dropout_rate': 0.44241272967497836, 'learning_rate': 0.0018302363254262002, 'batch_size': 16}. Best is trial 8 with value: 0.8911851558035646.


✅ Trial 9 finished with Mean F1-Score: 0.8678

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.8912

  Best Hyperparameters:
    f1_filters: 32
    depth_multiplier: 4
    dropout_rate: 0.341693680712148
    learning_rate: 0.0011756074074994365
    batch_size: 32


In [2]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, AveragePooling2D, Flatten,
                                     Dense, BatchNormalization, Activation, Dropout, SeparableConv2D)
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
input_shape = (27, 2560, 1)
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
all_data_folder = "Filtered_8_30Hz"

# --------------------------------------------------⚡️ Helper functions ---------------------------------------------------
def build_eegnet_model(input_shape, f1, d_mult, dropout_rate, lr, num_classes=1):
    """Builds the EEGNet model. The architecture is flexible to the new input shape."""
    inputs = Input(shape=input_shape)
    x = Conv2D(f1, (1, 64), padding='same', use_bias=False)(inputs)
    x = BatchNormalization()(x)
    # This DepthwiseConv2D layer is key; it performs the spatial convolution.
    # It correctly uses input_shape[0] (now 27) as the kernel height.
    x = DepthwiseConv2D((input_shape[0], 1), use_bias=False, depth_multiplier=d_mult, depthwise_constraint=max_norm(1.))(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropout_rate)(x)
    x = SeparableConv2D(f1 * d_mult, (1, 16), use_bias=False, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x)
    x = Dropout(dropout_rate)(x)
    x = Flatten()(x)
    x = Dense(num_classes, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=x)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

def load_data_from_folder(folder):
    """Loads file paths and labels from a specified folder as a Python list."""
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        """Use ceiling division to include the last, smaller batch."""
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        """Generates data containing batch_size samples."""
        data, labels = [], []
        for file, label in batch_data_list:
            sample = np.load(file) # Assumed shape: (2560, 27)
            
            # ✅ Transpose and reshape the data to match the new input_shape
            sample = sample.T # Transpose from (2560, 27) to (27, 2560)
            sample = np.expand_dims(sample, axis=-1) # Add channel dim to get (27, 2560, 1)
            
            data.append(sample)
            labels.append(label)

        # Stacking the list of correctly shaped samples creates the final batch
        data = np.array(data)
        labels = np.array(labels, dtype=np.float32)
        return data, labels

# --------------------------------------------------⚡️ Load all data ---------------------------------------------------
all_set = load_data_from_folder(all_data_folder)
print(f"📦 Total samples found: {len(all_set)}")
all_set = np.array(all_set, dtype=object)

# --------------------------------------------------⚡️ Objective Function for Optuna ---------------------------------------------------
def objective(trial):
    f1 = trial.suggest_categorical('f1_filters', [8, 16, 32])
    d_mult = trial.suggest_categorical('depth_multiplier', [2, 3, 4])
    dropout_rate = trial.suggest_float('dropout_rate', 0.25, 0.6)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    epochs = 70

    print(f"\n🚀 Starting Trial {trial.number} with params: F1={f1}, D_Mult={d_mult}, Dropout={dropout_rate:.3f}, LR={lr:.5f}, Batch={batch_size}")

    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_set)):
        print(f"--- Fold {fold + 1}/{n_splits} ---")
        train_data_list = all_set[train_idx]
        val_data_list = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data_list, batch_size=batch_size, shuffle=True)
        val_gen = EEGDataGenerator(val_data_list, batch_size=batch_size, shuffle=False)
        
        model = build_eegnet_model(input_shape, f1, d_mult, dropout_rate, lr)
        
        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=epochs,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # Get predictions from the generator
        y_pred_probs = model.predict(val_gen, verbose=0)
        
        # Extract true labels directly from the generator to guarantee alignment
        y_true = []
        for i in range(len(val_gen)):
            _, labels_batch = val_gen[i]
            y_true.extend(labels_batch)
        y_true = np.array(y_true)
        
        y_pred = (y_pred_probs > 0.5).astype(int)
        
        current_f1 = f1_score(y_true, y_pred)
        fold_f1_scores.append(current_f1)
        
    mean_f1_score = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1_score:.4f}")
    
    return mean_f1_score

# --------------------------------------------------⚡️ Run the Optuna Study ---------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# --------------------------------------------------⚡️ Print the results of the study ---------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")
print("\n🎉 Best trial:")
best_trial = study.best_trial
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}")
print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-10-19 04:27:30,736] A new study created in memory with name: no-name-462ccbbe-04eb-4854-96e6-d6cc6a5c3fb5


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: F1=16, D_Mult=2, Dropout=0.369, LR=0.00013, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-10-19 07:48:11,399] Trial 0 finished with value: 0.8630625476637352 and parameters: {'f1_filters': 16, 'depth_multiplier': 2, 'dropout_rate': 0.368708163953992, 'learning_rate': 0.00013356305634257726, 'batch_size': 16}. Best is trial 0 with value: 0.8630625476637352.


✅ Trial 0 finished with Mean F1-Score: 0.8631

🚀 Starting Trial 1 with params: F1=8, D_Mult=2, Dropout=0.301, LR=0.00242, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-10-19 09:39:59,552] Trial 1 finished with value: 0.8535955328555724 and parameters: {'f1_filters': 8, 'depth_multiplier': 2, 'dropout_rate': 0.3007816616148726, 'learning_rate': 0.0024246200050278425, 'batch_size': 16}. Best is trial 0 with value: 0.8630625476637352.


...

✅ Trial 7 finished with Mean F1-Score: 0.8603

🚀 Starting Trial 8 with params: F1=8, D_Mult=2, Dropout=0.582, LR=0.00070, Batch=64
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-10-20 01:23:06,253] Trial 8 finished with value: 0.8554235514611447 and parameters: {'f1_filters': 8, 'depth_multiplier': 2, 'dropout_rate': 0.5815011601473252, 'learning_rate': 0.0007030024207219794, 'batch_size': 64}. Best is trial 4 with value: 0.8658958594032334.


✅ Trial 8 finished with Mean F1-Score: 0.8554

🚀 Starting Trial 9 with params: F1=16, D_Mult=4, Dropout=0.381, LR=0.00071, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-10-20 02:55:11,190] Trial 9 finished with value: 0.8580876242933092 and parameters: {'f1_filters': 16, 'depth_multiplier': 4, 'dropout_rate': 0.38144424923511944, 'learning_rate': 0.0007076803743695869, 'batch_size': 32}. Best is trial 4 with value: 0.8658958594032334.


✅ Trial 9 finished with Mean F1-Score: 0.8581

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.8659

  Best Hyperparameters:
    f1_filters: 32
    depth_multiplier: 2
    dropout_rate: 0.41645535663828737
    learning_rate: 0.00017546079359303938
    batch_size: 32


In [1]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, AveragePooling2D, Flatten,
                                     Dense, BatchNormalization, Activation, Dropout, SeparableConv2D)
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
input_shape = (27, 2560, 1)
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
all_data_folder = "Filtered_8_13Hz"

# --------------------------------------------------⚡️ Helper functions ---------------------------------------------------
def build_eegnet_model(input_shape, f1, d_mult, dropout_rate, lr, num_classes=1):
    """Builds the EEGNet model. The architecture is flexible to the new input shape."""
    inputs = Input(shape=input_shape)
    x = Conv2D(f1, (1, 64), padding='same', use_bias=False)(inputs)
    x = BatchNormalization()(x)
    # This DepthwiseConv2D layer is key; it performs the spatial convolution.
    # It correctly uses input_shape[0] (now 27) as the kernel height.
    x = DepthwiseConv2D((input_shape[0], 1), use_bias=False, depth_multiplier=d_mult, depthwise_constraint=max_norm(1.))(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropout_rate)(x)
    x = SeparableConv2D(f1 * d_mult, (1, 16), use_bias=False, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x)
    x = Dropout(dropout_rate)(x)
    x = Flatten()(x)
    x = Dense(num_classes, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=x)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

def load_data_from_folder(folder):
    """Loads file paths and labels from a specified folder as a Python list."""
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        """Use ceiling division to include the last, smaller batch."""
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        """Generates data containing batch_size samples."""
        data, labels = [], []
        for file, label in batch_data_list:
            sample = np.load(file) # Assumed shape: (2560, 27)
            
            # ✅ Transpose and reshape the data to match the new input_shape
            sample = sample.T # Transpose from (2560, 27) to (27, 2560)
            sample = np.expand_dims(sample, axis=-1) # Add channel dim to get (27, 2560, 1)
            
            data.append(sample)
            labels.append(label)

        # Stacking the list of correctly shaped samples creates the final batch
        data = np.array(data)
        labels = np.array(labels, dtype=np.float32)
        return data, labels

# --------------------------------------------------⚡️ Load all data ---------------------------------------------------
all_set = load_data_from_folder(all_data_folder)
print(f"📦 Total samples found: {len(all_set)}")
all_set = np.array(all_set, dtype=object)

# --------------------------------------------------⚡️ Objective Function for Optuna ---------------------------------------------------
def objective(trial):
    f1 = trial.suggest_categorical('f1_filters', [8, 16, 32])
    d_mult = trial.suggest_categorical('depth_multiplier', [2, 3, 4])
    dropout_rate = trial.suggest_float('dropout_rate', 0.25, 0.6)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    epochs = 70

    print(f"\n🚀 Starting Trial {trial.number} with params: F1={f1}, D_Mult={d_mult}, Dropout={dropout_rate:.3f}, LR={lr:.5f}, Batch={batch_size}")

    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_set)):
        print(f"--- Fold {fold + 1}/{n_splits} ---")
        train_data_list = all_set[train_idx]
        val_data_list = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data_list, batch_size=batch_size, shuffle=True)
        val_gen = EEGDataGenerator(val_data_list, batch_size=batch_size, shuffle=False)
        
        model = build_eegnet_model(input_shape, f1, d_mult, dropout_rate, lr)
        
        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=epochs,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # Get predictions from the generator
        y_pred_probs = model.predict(val_gen, verbose=0)
        
        # Extract true labels directly from the generator to guarantee alignment
        y_true = []
        for i in range(len(val_gen)):
            _, labels_batch = val_gen[i]
            y_true.extend(labels_batch)
        y_true = np.array(y_true)
        
        y_pred = (y_pred_probs > 0.5).astype(int)
        
        current_f1 = f1_score(y_true, y_pred)
        fold_f1_scores.append(current_f1)
        
    mean_f1_score = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1_score:.4f}")
    
    return mean_f1_score

# --------------------------------------------------⚡️ Run the Optuna Study ---------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# --------------------------------------------------⚡️ Print the results of the study ---------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")
print("\n🎉 Best trial:")
best_trial = study.best_trial
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}")
print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-11-13 07:46:27,842] A new study created in memory with name: no-name-9cdbef5b-1bf5-4fa4-a554-bece9341a201


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: F1=16, D_Mult=4, Dropout=0.404, LR=0.00016, Batch=64
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-13 10:40:36,791] Trial 0 finished with value: 0.8593028130117926 and parameters: {'f1_filters': 16, 'depth_multiplier': 4, 'dropout_rate': 0.40389814257562723, 'learning_rate': 0.00016280775565318675, 'batch_size': 64}. Best is trial 0 with value: 0.8593028130117926.


✅ Trial 0 finished with Mean F1-Score: 0.8593

🚀 Starting Trial 1 with params: F1=16, D_Mult=3, Dropout=0.332, LR=0.00068, Batch=64
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-13 12:48:47,040] Trial 1 finished with value: 0.8650364789247236 and parameters: {'f1_filters': 16, 'depth_multiplier': 3, 'dropout_rate': 0.33198482892677056, 'learning_rate': 0.000677187628608627, 'batch_size': 64}. Best is trial 1 with value: 0.8650364789247236.


...

✅ Trial 7 finished with Mean F1-Score: 0.8551

🚀 Starting Trial 8 with params: F1=32, D_Mult=2, Dropout=0.461, LR=0.00023, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-14 06:07:07,413] Trial 8 finished with value: 0.8703919351949944 and parameters: {'f1_filters': 32, 'depth_multiplier': 2, 'dropout_rate': 0.46103561581931307, 'learning_rate': 0.00023117597342136787, 'batch_size': 32}. Best is trial 8 with value: 0.8703919351949944.


✅ Trial 8 finished with Mean F1-Score: 0.8704

🚀 Starting Trial 9 with params: F1=8, D_Mult=3, Dropout=0.537, LR=0.00029, Batch=64
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-14 09:40:52,919] Trial 9 finished with value: 0.857458891033114 and parameters: {'f1_filters': 8, 'depth_multiplier': 3, 'dropout_rate': 0.5369477183879604, 'learning_rate': 0.0002930343074698694, 'batch_size': 64}. Best is trial 8 with value: 0.8703919351949944.


✅ Trial 9 finished with Mean F1-Score: 0.8575

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.8704

  Best Hyperparameters:
    f1_filters: 32
    depth_multiplier: 2
    dropout_rate: 0.46103561581931307
    learning_rate: 0.00023117597342136787
    batch_size: 32


# CvT

🏆 Optuna Hyperparameter Optimization For Original Dataset

Best Hyperparameters:
- patch_size: 128
- projection_dim: 64
- num_heads: 4
- transformer_layers: 5
- ff_dim: 256
- dropout_rate: 0.2688835448060798
- learning_rate: 0.00019559517515984408
- batch_size: 32

🏆 Optuna Hyperparameter Optimization For Raw Dataset

Best Hyperparameters:
- patch_size: 128
- projection_dim: 128
- num_heads: 8
- transformer_layers: 4
- ff_dim: 128
- dropout_rate: 0.24669971969138374
- learning_rate: 0.00030491201360786663
- batch_size: 32

🏆 Optuna Hyperparameter Optimization For Filtered Dataset

Best Hyperparameters:
- patch_size: 128
- projection_dim: 128
- num_heads: 4
- transformer_layers: 4
- ff_dim: 128
- dropout_rate: 0.3387000649392182
- learning_rate: 0.00016084524900736826
- batch_size: 32

In [1]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
                                     Dropout, GlobalAveragePooling1D, 
                                     Conv1D, TimeDistributed)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
# Fixed settings for optimization
SEQ_LEN = 2560
INPUT_DIM = 27
N_SPLITS = 5
KF = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
ALL_DATA_FOLDER = "Original/All Data"
EPOCHS = 70

# --------------------------------------------------------⚡️ Custom Model Layers ---------------------------------------------------------
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

# ----------------------------------------------------------------------------------------------------
def get_sinusoidal_position_embedding(sequence_length, embedding_dim):
    """Generates the sinusoidal positional embedding matrix."""
    if embedding_dim % 2 != 0:
        raise ValueError(
            f"Cannot create sinusoidal embedding for odd dimension {embedding_dim}. "
            "Use an even dimension."
        )
            
    angle_rates = 1.0 / (10000 ** (
        tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)
    ))
    positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
    angle_rads = positions * angle_rates[tf.newaxis, :]
    sines = tf.math.sin(angle_rads)
    cosines = tf.math.cos(angle_rads)
    pos_encoding = tf.concat([sines, cosines], axis=-1)
    
    return pos_encoding # shape [sequence_length, embedding_dim]

# -----------------------------------⚡️ Function to build the CvT model (Accepts Hyperparameters) ------------------------------------
def build_eeg_transformer_vit_style(seq_len, input_dim, patch_size, projection_dim, num_heads,
                                      transformer_layers, ff_dim, dropout_rate, lr):
    
    # ✅ Important assertions to prevent architectural errors
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    assert projection_dim % num_heads == 0, "Projection dim must be divisible by number of heads."

    num_patches = seq_len // patch_size
    
    inputs = Input(shape=(seq_len, input_dim)) # Shape: [B, 2560, 27]
    patches = Patches(patch_size)(inputs) # Shape: [B, num_patches, patch_size, input_dim]
    
    cnn_patch_encoder = tf.keras.Sequential([
        Input(shape=(patch_size, input_dim)),
        Conv1D(filters=64, kernel_size=10, padding='same', activation='relu'),
        Conv1D(filters=64, kernel_size=5, padding='same', activation='relu'),
        GlobalAveragePooling1D(), # Shape: [B, 64]
        Dense(projection_dim)     # Shape: [B, projection_dim]
    ], name="cnn_patch_encoder")
    
    encoded_patches = TimeDistributed(cnn_patch_encoder)(patches)
    
    # ------------------
    # 3. Add Sinusoidal Position Embedding
    # ------------------
    pos_encoding_matrix = get_sinusoidal_position_embedding(num_patches, projection_dim)
    position_embedding = tf.Variable(
        pos_encoding_matrix,
        trainable=False, # This parameter is not learned
        name="position_embedding"
    )
    encoded_patches += position_embedding
    
    # ------------------
    # 4. Transformer Blocks
    # ------------------
    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    # ------------------
    # 5. Output Layers
    # ------------------
    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    outputs = Dense(1, activation='sigmoid')(outputs)
    
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------------------------------⚡️ Helper Functions -------------------------------------------------------
def load_data_from_folder(folder):
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, seq_len=2560, input_dim=27, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        data = np.zeros((len(batch_data_list), self.seq_len, self.input_dim))
        labels = np.zeros((len(batch_data_list),))
        
        for i, (file, label) in enumerate(batch_data_list):
            sample = np.load(file)
            if sample.shape == (self.input_dim, self.seq_len):
                sample = sample.T
            
            if sample.shape == (self.seq_len, self.input_dim):
                data[i,] = sample
                labels[i] = label
            else:
                print(f"Warning: Skipping file {file} with unexpected shape {sample.shape}")
        return data, labels

# ------------------------------------------------------⚡️ Load All Data ------------------------------------------------------
all_set = load_data_from_folder(ALL_DATA_FOLDER)
all_set = np.array(all_set, dtype=object)
print(f"📦 Total samples found: {len(all_set)}")

# --------------------------------------------⚡️ Objective Function for Optuna (F1-Score) --------------------------------------------
def objective(trial):
    # 1. Suggest hyperparameters for this trial
    patch_size = trial.suggest_categorical('patch_size', [128, 160, 256, 320])
    projection_dim = trial.suggest_categorical('projection_dim', [64, 128])
    num_heads = trial.suggest_categorical('num_heads', [4, 8])
    transformer_layers = trial.suggest_int('transformer_layers', 2, 6)
    ff_dim = trial.suggest_categorical('ff_dim', [128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    
    # If hyperparameters are invalid, prune the trial
    if projection_dim % num_heads != 0:
        raise optuna.exceptions.TrialPruned()
        
    # ✅ Ensure patch_size is compatible with SEQ_LEN
    if SEQ_LEN % patch_size != 0:
        print(f"Pruning trial: SEQ_LEN ({SEQ_LEN}) not divisible by patch_size ({patch_size})")
        raise optuna.exceptions.TrialPruned()


    print(f"\n🚀 Starting Trial {trial.number} with params: Patch={patch_size}, Proj_dim={projection_dim}, Heads={num_heads}, Layers={transformer_layers}, LR={lr:.5f}, Batch={batch_size}")
    
    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(KF.split(all_set)):
        print(f"--- Fold {fold + 1}/{N_SPLITS} ---")
        train_data = all_set[train_idx]
        val_data = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM)
        val_gen = EEGDataGenerator(val_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM, shuffle=False)
        
        model = build_eeg_transformer_vit_style(
            SEQ_LEN, INPUT_DIM, patch_size, projection_dim, num_heads,
            transformer_layers, ff_dim, dropout_rate, lr
        )

        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # ✅ Calculate F1-Score for evaluation
        y_pred_probs = model.predict(val_gen, verbose=0)
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()
        
        y_true = np.array([label for _, label in val_data])
        
        # Ensure vectors have the same length
        min_len = min(len(y_true), len(y_pred))
        
        if min_len == 0:
            print("Warning: Validation generator returned 0 samples. Skipping fold.")
            continue
            
        current_f1 = f1_score(y_true[:min_len], y_pred[:min_len])
        fold_f1_scores.append(current_f1)
        
    if not fold_f1_scores:
        print(f"Trial {trial.number} failed, no F1 scores recorded.")
        return 0.0 # Or another penalty value

    # ✅ Return the mean F1-Score for optimization
    mean_f1 = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1:.4f}")
    
    return mean_f1

# ------------------------------------------------------⚡️ Run the Optuna Study ------------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# -------------------------------------------------⚡️ Print the results of the study -------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial
print("\n🎉 Best trial:")
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}") # ✅ Display the best F1-Score

print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-11-03 14:59:30,223] A new study created in memory with name: no-name-37030179-983d-48bf-9177-fb3caa9850d0


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: Patch=320, Proj_dim=64, Heads=4, Layers=3, LR=0.00012, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-03 16:26:21,619] Trial 0 finished with value: 0.7134413058785768 and parameters: {'patch_size': 320, 'projection_dim': 64, 'num_heads': 4, 'transformer_layers': 3, 'ff_dim': 128, 'dropout_rate': 0.2213604451832296, 'learning_rate': 0.00012037039238301257, 'batch_size': 16}. Best is trial 0 with value: 0.7134413058785768.


✅ Trial 0 finished with Mean F1-Score: 0.7134

🚀 Starting Trial 1 with params: Patch=160, Proj_dim=128, Heads=8, Layers=2, LR=0.00012, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-03 17:34:40,757] Trial 1 finished with value: 0.7334820334282008 and parameters: {'patch_size': 160, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 2, 'ff_dim': 128, 'dropout_rate': 0.14421695454973493, 'learning_rate': 0.00012152810268962299, 'batch_size': 32}. Best is trial 1 with value: 0.7334820334282008.


...

✅ Trial 7 finished with Mean F1-Score: 0.5936

🚀 Starting Trial 8 with params: Patch=128, Proj_dim=64, Heads=4, Layers=5, LR=0.00020, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-04 04:07:18,714] Trial 8 finished with value: 0.7362470301180667 and parameters: {'patch_size': 128, 'projection_dim': 64, 'num_heads': 4, 'transformer_layers': 5, 'ff_dim': 256, 'dropout_rate': 0.2688835448060798, 'learning_rate': 0.00019559517515984408, 'batch_size': 32}. Best is trial 8 with value: 0.7362470301180667.


✅ Trial 8 finished with Mean F1-Score: 0.7362

🚀 Starting Trial 9 with params: Patch=128, Proj_dim=128, Heads=8, Layers=4, LR=0.00081, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-04 05:43:37,655] Trial 9 finished with value: 0.5685831185731163 and parameters: {'patch_size': 128, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 4, 'ff_dim': 128, 'dropout_rate': 0.14485267115362693, 'learning_rate': 0.0008082211540101703, 'batch_size': 32}. Best is trial 8 with value: 0.7362470301180667.


✅ Trial 9 finished with Mean F1-Score: 0.5686

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.7362

  Best Hyperparameters:
    patch_size: 128
    projection_dim: 64
    num_heads: 4
    transformer_layers: 5
    ff_dim: 256
    dropout_rate: 0.2688835448060798
    learning_rate: 0.00019559517515984408
    batch_size: 32


In [2]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
                                     Dropout, GlobalAveragePooling1D, 
                                     Conv1D, TimeDistributed)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
# Fixed settings for optimization
SEQ_LEN = 2560
INPUT_DIM = 27
N_SPLITS = 5
KF = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
ALL_DATA_FOLDER = "Raw/All Data"
EPOCHS = 70

# --------------------------------------------------------⚡️ Custom Model Layers ---------------------------------------------------------
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

# ----------------------------------------------------------------------------------------------------
def get_sinusoidal_position_embedding(sequence_length, embedding_dim):
    """Generates the sinusoidal positional embedding matrix."""
    if embedding_dim % 2 != 0:
        raise ValueError(
            f"Cannot create sinusoidal embedding for odd dimension {embedding_dim}. "
            "Use an even dimension."
        )
            
    angle_rates = 1.0 / (10000 ** (
        tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)
    ))
    positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
    angle_rads = positions * angle_rates[tf.newaxis, :]
    sines = tf.math.sin(angle_rads)
    cosines = tf.math.cos(angle_rads)
    pos_encoding = tf.concat([sines, cosines], axis=-1)
    
    return pos_encoding # shape [sequence_length, embedding_dim]

# -----------------------------------⚡️ Function to build the CvT model (Accepts Hyperparameters) ------------------------------------
def build_eeg_transformer_vit_style(seq_len, input_dim, patch_size, projection_dim, num_heads,
                                      transformer_layers, ff_dim, dropout_rate, lr):
    
    # ✅ Important assertions to prevent architectural errors
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    assert projection_dim % num_heads == 0, "Projection dim must be divisible by number of heads."

    num_patches = seq_len // patch_size
    
    inputs = Input(shape=(seq_len, input_dim)) # Shape: [B, 2560, 27]
    patches = Patches(patch_size)(inputs) # Shape: [B, num_patches, patch_size, input_dim]
    
    cnn_patch_encoder = tf.keras.Sequential([
        Input(shape=(patch_size, input_dim)),
        Conv1D(filters=64, kernel_size=10, padding='same', activation='relu'),
        Conv1D(filters=64, kernel_size=5, padding='same', activation='relu'),
        GlobalAveragePooling1D(), # Shape: [B, 64]
        Dense(projection_dim)     # Shape: [B, projection_dim]
    ], name="cnn_patch_encoder")
    
    encoded_patches = TimeDistributed(cnn_patch_encoder)(patches)
    
    # ------------------
    # 3. Add Sinusoidal Position Embedding
    # ------------------
    pos_encoding_matrix = get_sinusoidal_position_embedding(num_patches, projection_dim)
    position_embedding = tf.Variable(
        pos_encoding_matrix,
        trainable=False, # This parameter is not learned
        name="position_embedding"
    )
    encoded_patches += position_embedding
    
    # ------------------
    # 4. Transformer Blocks
    # ------------------
    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    # ------------------
    # 5. Output Layers
    # ------------------
    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    outputs = Dense(1, activation='sigmoid')(outputs)
    
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------------------------------⚡️ Helper Functions -------------------------------------------------------
def load_data_from_folder(folder):
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, seq_len=2560, input_dim=27, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        data = np.zeros((len(batch_data_list), self.seq_len, self.input_dim))
        labels = np.zeros((len(batch_data_list),))
        
        for i, (file, label) in enumerate(batch_data_list):
            sample = np.load(file)
            if sample.shape == (self.input_dim, self.seq_len):
                sample = sample.T
            
            if sample.shape == (self.seq_len, self.input_dim):
                data[i,] = sample
                labels[i] = label
            else:
                print(f"Warning: Skipping file {file} with unexpected shape {sample.shape}")
        return data, labels

# ------------------------------------------------------⚡️ Load All Data ------------------------------------------------------
all_set = load_data_from_folder(ALL_DATA_FOLDER)
all_set = np.array(all_set, dtype=object)
print(f"📦 Total samples found: {len(all_set)}")

# --------------------------------------------⚡️ Objective Function for Optuna (F1-Score) --------------------------------------------
def objective(trial):
    # 1. Suggest hyperparameters for this trial
    patch_size = trial.suggest_categorical('patch_size', [128, 160, 256, 320])
    projection_dim = trial.suggest_categorical('projection_dim', [64, 128])
    num_heads = trial.suggest_categorical('num_heads', [4, 8])
    transformer_layers = trial.suggest_int('transformer_layers', 2, 6)
    ff_dim = trial.suggest_categorical('ff_dim', [128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    
    # If hyperparameters are invalid, prune the trial
    if projection_dim % num_heads != 0:
        raise optuna.exceptions.TrialPruned()
        
    # ✅ Ensure patch_size is compatible with SEQ_LEN
    if SEQ_LEN % patch_size != 0:
        print(f"Pruning trial: SEQ_LEN ({SEQ_LEN}) not divisible by patch_size ({patch_size})")
        raise optuna.exceptions.TrialPruned()


    print(f"\n🚀 Starting Trial {trial.number} with params: Patch={patch_size}, Proj_dim={projection_dim}, Heads={num_heads}, Layers={transformer_layers}, LR={lr:.5f}, Batch={batch_size}")
    
    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(KF.split(all_set)):
        print(f"--- Fold {fold + 1}/{N_SPLITS} ---")
        train_data = all_set[train_idx]
        val_data = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM)
        val_gen = EEGDataGenerator(val_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM, shuffle=False)
        
        model = build_eeg_transformer_vit_style(
            SEQ_LEN, INPUT_DIM, patch_size, projection_dim, num_heads,
            transformer_layers, ff_dim, dropout_rate, lr
        )

        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # ✅ Calculate F1-Score for evaluation
        y_pred_probs = model.predict(val_gen, verbose=0)
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()
        
        y_true = np.array([label for _, label in val_data])
        
        # Ensure vectors have the same length
        min_len = min(len(y_true), len(y_pred))
        
        if min_len == 0:
            print("Warning: Validation generator returned 0 samples. Skipping fold.")
            continue
            
        current_f1 = f1_score(y_true[:min_len], y_pred[:min_len])
        fold_f1_scores.append(current_f1)
        
    if not fold_f1_scores:
        print(f"Trial {trial.number} failed, no F1 scores recorded.")
        return 0.0 # Or another penalty value

    # ✅ Return the mean F1-Score for optimization
    mean_f1 = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1:.4f}")
    
    return mean_f1

# ------------------------------------------------------⚡️ Run the Optuna Study ------------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# -------------------------------------------------⚡️ Print the results of the study -------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial
print("\n🎉 Best trial:")
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}") # ✅ Display the best F1-Score

print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-11-04 05:43:38,416] A new study created in memory with name: no-name-468afaa4-bd47-419d-963d-a5ace75f916c


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: Patch=128, Proj_dim=128, Heads=8, Layers=5, LR=0.00037, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-04 07:44:13,848] Trial 0 finished with value: 0.8027919111093954 and parameters: {'patch_size': 128, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 5, 'ff_dim': 128, 'dropout_rate': 0.2582721854110099, 'learning_rate': 0.0003711572876366871, 'batch_size': 32}. Best is trial 0 with value: 0.8027919111093954.


✅ Trial 0 finished with Mean F1-Score: 0.8028

🚀 Starting Trial 1 with params: Patch=160, Proj_dim=64, Heads=4, Layers=3, LR=0.00032, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-04 09:43:35,636] Trial 1 finished with value: 0.7545001675321382 and parameters: {'patch_size': 160, 'projection_dim': 64, 'num_heads': 4, 'transformer_layers': 3, 'ff_dim': 256, 'dropout_rate': 0.2702020109476313, 'learning_rate': 0.00031812331960520647, 'batch_size': 32}. Best is trial 0 with value: 0.8027919111093954.


...

✅ Trial 7 finished with Mean F1-Score: 0.7031

🚀 Starting Trial 8 with params: Patch=256, Proj_dim=128, Heads=4, Layers=4, LR=0.00067, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-05 08:48:20,628] Trial 8 finished with value: 0.711952596446566 and parameters: {'patch_size': 256, 'projection_dim': 128, 'num_heads': 4, 'transformer_layers': 4, 'ff_dim': 128, 'dropout_rate': 0.24584121882605717, 'learning_rate': 0.0006744961656985228, 'batch_size': 16}. Best is trial 4 with value: 0.8053227781678215.


✅ Trial 8 finished with Mean F1-Score: 0.7120

🚀 Starting Trial 9 with params: Patch=128, Proj_dim=128, Heads=4, Layers=6, LR=0.00061, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-05 18:57:32,672] Trial 9 finished with value: 0.7130054642584318 and parameters: {'patch_size': 128, 'projection_dim': 128, 'num_heads': 4, 'transformer_layers': 6, 'ff_dim': 128, 'dropout_rate': 0.13323872977944512, 'learning_rate': 0.0006090156878595406, 'batch_size': 16}. Best is trial 4 with value: 0.8053227781678215.


✅ Trial 9 finished with Mean F1-Score: 0.7130

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.8053

  Best Hyperparameters:
    patch_size: 128
    projection_dim: 128
    num_heads: 8
    transformer_layers: 4
    ff_dim: 128
    dropout_rate: 0.24669971969138374
    learning_rate: 0.00030491201360786663
    batch_size: 32


In [1]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
                                     Dropout, GlobalAveragePooling1D, 
                                     Conv1D, TimeDistributed)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
# Fixed settings for optimization
SEQ_LEN = 2560
INPUT_DIM = 27
N_SPLITS = 5
KF = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
ALL_DATA_FOLDER = "Filtered_8_30Hz"
EPOCHS = 70

# --------------------------------------------------------⚡️ Custom Model Layers ---------------------------------------------------------
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

# ----------------------------------------------------------------------------------------------------
def get_sinusoidal_position_embedding(sequence_length, embedding_dim):
    """Generates the sinusoidal positional embedding matrix."""
    if embedding_dim % 2 != 0:
        raise ValueError(
            f"Cannot create sinusoidal embedding for odd dimension {embedding_dim}. "
            "Use an even dimension."
        )
            
    angle_rates = 1.0 / (10000 ** (
        tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)
    ))
    positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
    angle_rads = positions * angle_rates[tf.newaxis, :]
    sines = tf.math.sin(angle_rads)
    cosines = tf.math.cos(angle_rads)
    pos_encoding = tf.concat([sines, cosines], axis=-1)
    
    return pos_encoding # shape [sequence_length, embedding_dim]

# -----------------------------------⚡️ Function to build the CvT model (Accepts Hyperparameters) ------------------------------------
def build_eeg_transformer_vit_style(seq_len, input_dim, patch_size, projection_dim, num_heads,
                                      transformer_layers, ff_dim, dropout_rate, lr):
    
    # ✅ Important assertions to prevent architectural errors
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    assert projection_dim % num_heads == 0, "Projection dim must be divisible by number of heads."

    num_patches = seq_len // patch_size
    
    inputs = Input(shape=(seq_len, input_dim)) # Shape: [B, 2560, 27]
    patches = Patches(patch_size)(inputs) # Shape: [B, num_patches, patch_size, input_dim]
    
    cnn_patch_encoder = tf.keras.Sequential([
        Input(shape=(patch_size, input_dim)),
        Conv1D(filters=64, kernel_size=10, padding='same', activation='relu'),
        Conv1D(filters=64, kernel_size=5, padding='same', activation='relu'),
        GlobalAveragePooling1D(), # Shape: [B, 64]
        Dense(projection_dim)     # Shape: [B, projection_dim]
    ], name="cnn_patch_encoder")
    
    encoded_patches = TimeDistributed(cnn_patch_encoder)(patches)
    
    # ------------------
    # 3. Add Sinusoidal Position Embedding
    # ------------------
    pos_encoding_matrix = get_sinusoidal_position_embedding(num_patches, projection_dim)
    position_embedding = tf.Variable(
        pos_encoding_matrix,
        trainable=False, # This parameter is not learned
        name="position_embedding"
    )
    encoded_patches += position_embedding
    
    # ------------------
    # 4. Transformer Blocks
    # ------------------
    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    # ------------------
    # 5. Output Layers
    # ------------------
    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    outputs = Dense(1, activation='sigmoid')(outputs)
    
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------------------------------⚡️ Helper Functions -------------------------------------------------------
def load_data_from_folder(folder):
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, seq_len=2560, input_dim=27, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        data = np.zeros((len(batch_data_list), self.seq_len, self.input_dim))
        labels = np.zeros((len(batch_data_list),))
        
        for i, (file, label) in enumerate(batch_data_list):
            sample = np.load(file)
            if sample.shape == (self.input_dim, self.seq_len):
                sample = sample.T
            
            if sample.shape == (self.seq_len, self.input_dim):
                data[i,] = sample
                labels[i] = label
            else:
                print(f"Warning: Skipping file {file} with unexpected shape {sample.shape}")
        return data, labels

# ------------------------------------------------------⚡️ Load All Data ------------------------------------------------------
all_set = load_data_from_folder(ALL_DATA_FOLDER)
all_set = np.array(all_set, dtype=object)
print(f"📦 Total samples found: {len(all_set)}")

# --------------------------------------------⚡️ Objective Function for Optuna (F1-Score) --------------------------------------------
def objective(trial):
    # 1. Suggest hyperparameters for this trial
    patch_size = trial.suggest_categorical('patch_size', [128, 160, 256, 320])
    projection_dim = trial.suggest_categorical('projection_dim', [64, 128])
    num_heads = trial.suggest_categorical('num_heads', [4, 8])
    transformer_layers = trial.suggest_int('transformer_layers', 2, 6)
    ff_dim = trial.suggest_categorical('ff_dim', [128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    
    # If hyperparameters are invalid, prune the trial
    if projection_dim % num_heads != 0:
        raise optuna.exceptions.TrialPruned()
        
    # ✅ Ensure patch_size is compatible with SEQ_LEN
    if SEQ_LEN % patch_size != 0:
        print(f"Pruning trial: SEQ_LEN ({SEQ_LEN}) not divisible by patch_size ({patch_size})")
        raise optuna.exceptions.TrialPruned()


    print(f"\n🚀 Starting Trial {trial.number} with params: Patch={patch_size}, Proj_dim={projection_dim}, Heads={num_heads}, Layers={transformer_layers}, LR={lr:.5f}, Batch={batch_size}")
    
    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(KF.split(all_set)):
        print(f"--- Fold {fold + 1}/{N_SPLITS} ---")
        train_data = all_set[train_idx]
        val_data = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM)
        val_gen = EEGDataGenerator(val_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM, shuffle=False)
        
        model = build_eeg_transformer_vit_style(
            SEQ_LEN, INPUT_DIM, patch_size, projection_dim, num_heads,
            transformer_layers, ff_dim, dropout_rate, lr
        )

        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # ✅ Calculate F1-Score for evaluation
        y_pred_probs = model.predict(val_gen, verbose=0)
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()
        
        y_true = np.array([label for _, label in val_data])
        
        # Ensure vectors have the same length
        min_len = min(len(y_true), len(y_pred))
        
        if min_len == 0:
            print("Warning: Validation generator returned 0 samples. Skipping fold.")
            continue
            
        current_f1 = f1_score(y_true[:min_len], y_pred[:min_len])
        fold_f1_scores.append(current_f1)
        
    if not fold_f1_scores:
        print(f"Trial {trial.number} failed, no F1 scores recorded.")
        return 0.0 # Or another penalty value

    # ✅ Return the mean F1-Score for optimization
    mean_f1 = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1:.4f}")
    
    return mean_f1

# ------------------------------------------------------⚡️ Run the Optuna Study ------------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# -------------------------------------------------⚡️ Print the results of the study -------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial
print("\n🎉 Best trial:")
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}") # ✅ Display the best F1-Score

print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-11-08 17:29:22,675] A new study created in memory with name: no-name-7e4cc2a6-cbf7-4faf-b56b-a52c26f0e44e


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: Patch=320, Proj_dim=64, Heads=8, Layers=2, LR=0.00027, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-08 19:28:55,778] Trial 0 finished with value: 0.7758237910236547 and parameters: {'patch_size': 320, 'projection_dim': 64, 'num_heads': 8, 'transformer_layers': 2, 'ff_dim': 128, 'dropout_rate': 0.24377329796932495, 'learning_rate': 0.0002690304291822204, 'batch_size': 16}. Best is trial 0 with value: 0.7758237910236547.


✅ Trial 0 finished with Mean F1-Score: 0.7758

🚀 Starting Trial 1 with params: Patch=128, Proj_dim=128, Heads=4, Layers=6, LR=0.00044, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-08 21:39:37,063] Trial 1 finished with value: 0.797827454381307 and parameters: {'patch_size': 128, 'projection_dim': 128, 'num_heads': 4, 'transformer_layers': 6, 'ff_dim': 128, 'dropout_rate': 0.2714402689011841, 'learning_rate': 0.00044006081396430084, 'batch_size': 32}. Best is trial 1 with value: 0.797827454381307.


...

✅ Trial 7 finished with Mean F1-Score: 0.7997

🚀 Starting Trial 8 with params: Patch=320, Proj_dim=64, Heads=4, Layers=3, LR=0.00027, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-09 10:13:44,514] Trial 8 finished with value: 0.7728828444706576 and parameters: {'patch_size': 320, 'projection_dim': 64, 'num_heads': 4, 'transformer_layers': 3, 'ff_dim': 256, 'dropout_rate': 0.36425697685364866, 'learning_rate': 0.00026839999340950174, 'batch_size': 32}. Best is trial 7 with value: 0.7997317060959124.


✅ Trial 8 finished with Mean F1-Score: 0.7729

🚀 Starting Trial 9 with params: Patch=128, Proj_dim=64, Heads=8, Layers=5, LR=0.00014, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-09 12:32:06,057] Trial 9 finished with value: 0.7948192321385148 and parameters: {'patch_size': 128, 'projection_dim': 64, 'num_heads': 8, 'transformer_layers': 5, 'ff_dim': 128, 'dropout_rate': 0.10161211195142983, 'learning_rate': 0.0001362569244132083, 'batch_size': 16}. Best is trial 7 with value: 0.7997317060959124.


✅ Trial 9 finished with Mean F1-Score: 0.7948

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.7997

  Best Hyperparameters:
    patch_size: 128
    projection_dim: 128
    num_heads: 4
    transformer_layers: 4
    ff_dim: 128
    dropout_rate: 0.3387000649392182
    learning_rate: 0.00016084524900736826
    batch_size: 32


In [2]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
                                     Dropout, GlobalAveragePooling1D, 
                                     Conv1D, TimeDistributed)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
# Fixed settings for optimization
SEQ_LEN = 2560
INPUT_DIM = 27
N_SPLITS = 5
KF = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
ALL_DATA_FOLDER = "Filtered_8_13Hz"
EPOCHS = 70

# --------------------------------------------------------⚡️ Custom Model Layers ---------------------------------------------------------
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

# ----------------------------------------------------------------------------------------------------
def get_sinusoidal_position_embedding(sequence_length, embedding_dim):
    """Generates the sinusoidal positional embedding matrix."""
    if embedding_dim % 2 != 0:
        raise ValueError(
            f"Cannot create sinusoidal embedding for odd dimension {embedding_dim}. "
            "Use an even dimension."
        )
            
    angle_rates = 1.0 / (10000 ** (
        tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)
    ))
    positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
    angle_rads = positions * angle_rates[tf.newaxis, :]
    sines = tf.math.sin(angle_rads)
    cosines = tf.math.cos(angle_rads)
    pos_encoding = tf.concat([sines, cosines], axis=-1)
    
    return pos_encoding # shape [sequence_length, embedding_dim]

# -----------------------------------⚡️ Function to build the CvT model (Accepts Hyperparameters) ------------------------------------
def build_eeg_transformer_vit_style(seq_len, input_dim, patch_size, projection_dim, num_heads,
                                      transformer_layers, ff_dim, dropout_rate, lr):
    
    # ✅ Important assertions to prevent architectural errors
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    assert projection_dim % num_heads == 0, "Projection dim must be divisible by number of heads."

    num_patches = seq_len // patch_size
    
    inputs = Input(shape=(seq_len, input_dim)) # Shape: [B, 2560, 27]
    patches = Patches(patch_size)(inputs) # Shape: [B, num_patches, patch_size, input_dim]
    
    cnn_patch_encoder = tf.keras.Sequential([
        Input(shape=(patch_size, input_dim)),
        Conv1D(filters=64, kernel_size=10, padding='same', activation='relu'),
        Conv1D(filters=64, kernel_size=5, padding='same', activation='relu'),
        GlobalAveragePooling1D(), # Shape: [B, 64]
        Dense(projection_dim)     # Shape: [B, projection_dim]
    ], name="cnn_patch_encoder")
    
    encoded_patches = TimeDistributed(cnn_patch_encoder)(patches)
    
    # ------------------
    # 3. Add Sinusoidal Position Embedding
    # ------------------
    pos_encoding_matrix = get_sinusoidal_position_embedding(num_patches, projection_dim)
    position_embedding = tf.Variable(
        pos_encoding_matrix,
        trainable=False, # This parameter is not learned
        name="position_embedding"
    )
    encoded_patches += position_embedding
    
    # ------------------
    # 4. Transformer Blocks
    # ------------------
    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    # ------------------
    # 5. Output Layers
    # ------------------
    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    outputs = Dense(1, activation='sigmoid')(outputs)
    
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------------------------------⚡️ Helper Functions -------------------------------------------------------
def load_data_from_folder(folder):
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, seq_len=2560, input_dim=27, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        data = np.zeros((len(batch_data_list), self.seq_len, self.input_dim))
        labels = np.zeros((len(batch_data_list),))
        
        for i, (file, label) in enumerate(batch_data_list):
            sample = np.load(file)
            if sample.shape == (self.input_dim, self.seq_len):
                sample = sample.T
            
            if sample.shape == (self.seq_len, self.input_dim):
                data[i,] = sample
                labels[i] = label
            else:
                print(f"Warning: Skipping file {file} with unexpected shape {sample.shape}")
        return data, labels

# ------------------------------------------------------⚡️ Load All Data ------------------------------------------------------
all_set = load_data_from_folder(ALL_DATA_FOLDER)
all_set = np.array(all_set, dtype=object)
print(f"📦 Total samples found: {len(all_set)}")

# --------------------------------------------⚡️ Objective Function for Optuna (F1-Score) --------------------------------------------
def objective(trial):
    # 1. Suggest hyperparameters for this trial
    patch_size = trial.suggest_categorical('patch_size', [128, 160, 256, 320])
    projection_dim = trial.suggest_categorical('projection_dim', [64, 128])
    num_heads = trial.suggest_categorical('num_heads', [4, 8])
    transformer_layers = trial.suggest_int('transformer_layers', 2, 6)
    ff_dim = trial.suggest_categorical('ff_dim', [128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    
    # If hyperparameters are invalid, prune the trial
    if projection_dim % num_heads != 0:
        raise optuna.exceptions.TrialPruned()
        
    # ✅ Ensure patch_size is compatible with SEQ_LEN
    if SEQ_LEN % patch_size != 0:
        print(f"Pruning trial: SEQ_LEN ({SEQ_LEN}) not divisible by patch_size ({patch_size})")
        raise optuna.exceptions.TrialPruned()


    print(f"\n🚀 Starting Trial {trial.number} with params: Patch={patch_size}, Proj_dim={projection_dim}, Heads={num_heads}, Layers={transformer_layers}, LR={lr:.5f}, Batch={batch_size}")
    
    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(KF.split(all_set)):
        print(f"--- Fold {fold + 1}/{N_SPLITS} ---")
        train_data = all_set[train_idx]
        val_data = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM)
        val_gen = EEGDataGenerator(val_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM, shuffle=False)
        
        model = build_eeg_transformer_vit_style(
            SEQ_LEN, INPUT_DIM, patch_size, projection_dim, num_heads,
            transformer_layers, ff_dim, dropout_rate, lr
        )

        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # ✅ Calculate F1-Score for evaluation
        y_pred_probs = model.predict(val_gen, verbose=0)
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()
        
        y_true = np.array([label for _, label in val_data])
        
        # Ensure vectors have the same length
        min_len = min(len(y_true), len(y_pred))
        
        if min_len == 0:
            print("Warning: Validation generator returned 0 samples. Skipping fold.")
            continue
            
        current_f1 = f1_score(y_true[:min_len], y_pred[:min_len])
        fold_f1_scores.append(current_f1)
        
    if not fold_f1_scores:
        print(f"Trial {trial.number} failed, no F1 scores recorded.")
        return 0.0 # Or another penalty value

    # ✅ Return the mean F1-Score for optimization
    mean_f1 = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1:.4f}")
    
    return mean_f1

# ------------------------------------------------------⚡️ Run the Optuna Study ------------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# -------------------------------------------------⚡️ Print the results of the study -------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial
print("\n🎉 Best trial:")
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}") # ✅ Display the best F1-Score

print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-11-14 09:40:53,059] A new study created in memory with name: no-name-298bec45-ee23-46fc-86fd-9f671e4e3152


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: Patch=320, Proj_dim=64, Heads=4, Layers=4, LR=0.00023, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-14 12:13:25,131] Trial 0 finished with value: 0.7858955590786076 and parameters: {'patch_size': 320, 'projection_dim': 64, 'num_heads': 4, 'transformer_layers': 4, 'ff_dim': 256, 'dropout_rate': 0.36256383751279175, 'learning_rate': 0.00022604010854894714, 'batch_size': 32}. Best is trial 0 with value: 0.7858955590786076.


✅ Trial 0 finished with Mean F1-Score: 0.7859

🚀 Starting Trial 1 with params: Patch=256, Proj_dim=64, Heads=4, Layers=5, LR=0.00045, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-14 14:42:32,560] Trial 1 finished with value: 0.7922327119070955 and parameters: {'patch_size': 256, 'projection_dim': 64, 'num_heads': 4, 'transformer_layers': 5, 'ff_dim': 256, 'dropout_rate': 0.3380075492263521, 'learning_rate': 0.000449267704662728, 'batch_size': 32}. Best is trial 1 with value: 0.7922327119070955.


...

✅ Trial 7 finished with Mean F1-Score: 0.8043

🚀 Starting Trial 8 with params: Patch=256, Proj_dim=128, Heads=8, Layers=6, LR=0.00038, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-15 14:08:33,280] Trial 8 finished with value: 0.793352172132651 and parameters: {'patch_size': 256, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 6, 'ff_dim': 128, 'dropout_rate': 0.37980474101891715, 'learning_rate': 0.00037889842890979187, 'batch_size': 16}. Best is trial 2 with value: 0.8113523185881084.


✅ Trial 8 finished with Mean F1-Score: 0.7934

🚀 Starting Trial 9 with params: Patch=160, Proj_dim=128, Heads=8, Layers=2, LR=0.00087, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-15 15:35:13,796] Trial 9 finished with value: 0.8029713794991125 and parameters: {'patch_size': 160, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 2, 'ff_dim': 256, 'dropout_rate': 0.1113578642684607, 'learning_rate': 0.0008746164271622874, 'batch_size': 32}. Best is trial 2 with value: 0.8113523185881084.


✅ Trial 9 finished with Mean F1-Score: 0.8030

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.8114

  Best Hyperparameters:
    patch_size: 160
    projection_dim: 128
    num_heads: 8
    transformer_layers: 5
    ff_dim: 128
    dropout_rate: 0.19138784546099874
    learning_rate: 0.0003152536798449043
    batch_size: 32


# EEG Transformer Sinusoidal PE

🏆 Optuna Hyperparameter Optimization For Original Dataset

Best Hyperparameters:
- patch_size: 320
- projection_dim: 128
- num_heads: 8
- transformer_layers: 3
- ff_dim: 256
- dropout_rate: 0.3755571227807356
- learning_rate: 0.00018907008232065594
- batch_size: 32

🏆 Optuna Hyperparameter Optimization For Raw Dataset

Best Hyperparameters:
- patch_size: 320
- projection_dim: 128
- num_heads: 4
- transformer_layers: 2
- ff_dim: 256
- dropout_rate: 0.3409713547639843
- learning_rate: 0.00019259454444870477
- batch_size: 32

🏆 Optuna Hyperparameter Optimization For Filtered Dataset

Best Hyperparameters:
- patch_size: 128
- projection_dim: 128
- num_heads: 8
- transformer_layers: 2
- ff_dim: 128
- dropout_rate: 0.36094771336885223
- learning_rate: 0.0001730522300521005
- batch_size: 32

In [2]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
                                     Dropout, GlobalAveragePooling1D, Embedding)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
# Fixed settings for optimization
SEQ_LEN = 2560
INPUT_DIM = 27
N_SPLITS = 5
KF = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
ALL_DATA_FOLDER = "Original/All Data"
EPOCHS = 70

# --------------------------------------------------------⚡️ Custom Model Layers ---------------------------------------------------------
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

class PatchEncoder(Layer):
    """
    Encodes each EEG patch into a vector and adds a fixed Sinusoidal position embedding.
    """
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = Dense(units=projection_dim)
        
        # --- ✅ Create the fixed Sinusoidal Positional Embedding ---
        pos_encoding_matrix = self.get_sinusoidal_position_embedding(
            num_patches, projection_dim
        )
        self.position_embedding = tf.Variable(
            pos_encoding_matrix,
            trainable=False, # This parameter is not learned
            name="position_embedding"
        )

    def get_sinusoidal_position_embedding(self, sequence_length, embedding_dim):
        """Generates the sinusoidal positional embedding matrix."""
        if embedding_dim % 2 != 0:
            raise ValueError(
                f"Cannot create sinusoidal embedding for odd dimension {embedding_dim}. "
                "Use an even dimension."
            )
            
        angle_rates = 1.0 / (10000 ** (
            tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)
        ))
        positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
        angle_rads = positions * angle_rates[tf.newaxis, :]
        sines = tf.math.sin(angle_rads)
        cosines = tf.math.cos(angle_rads)
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        
        return pos_encoding # shape [sequence_length, embedding_dim]

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        patch_height = patch.shape[-2]
        patch_width = patch.shape[-1]
        
        patch_flattened = tf.reshape(patch, [batch_size, self.num_patches, patch_height * patch_width])
        encoded = self.projection(patch_flattened)
        
        # --- ✅ Add the fixed embedding matrix ---
        encoded += self.position_embedding 
        
        return encoded

    def get_config(self):
        config = super(PatchEncoder, self).get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config

# -----------------------------------⚡️ Function to build the ViT model (Accepts Hyperparameters) ------------------------------------
# ✅ This function now automatically uses the new PatchEncoder
def build_eeg_transformer_vit_style(seq_len, input_dim, patch_size, projection_dim, num_heads,
                                     transformer_layers, ff_dim, dropout_rate, lr):
    # ✅ Important assertions to prevent architectural errors
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    assert projection_dim % num_heads == 0, "Projection dim must be divisible by number of heads."

    num_patches = seq_len // patch_size
    inputs = Input(shape=(seq_len, input_dim))
    
    patches = Patches(patch_size)(inputs)
    # ✅ This line now uses the Sinusoidal PatchEncoder class defined above
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    outputs = Dense(1, activation='sigmoid')(outputs)
    
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------------------------------⚡️ Helper Functions -------------------------------------------------------
def load_data_from_folder(folder):
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, seq_len=2560, input_dim=27, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        data = np.zeros((len(batch_data_list), self.seq_len, self.input_dim))
        labels = np.zeros((len(batch_data_list),))
        
        for i, (file, label) in enumerate(batch_data_list):
            sample = np.load(file)
            # This model requires the shape (signal_length, num_channels)
            # ✅ Ensure correct data shape based on your code
            if sample.shape == (self.input_dim, self.seq_len):
                sample = sample.T
            
            if sample.shape == (self.seq_len, self.input_dim):
                data[i,] = sample
                labels[i] = label
            else:
                # If data doesn't have the expected shape, log it
                print(f"Warning: Skipping file {file} with unexpected shape {sample.shape}")

        # Filtering samples that might not have been loaded (though this won't happen with np.zeros)
        return data, labels

# ------------------------------------------------------⚡️ Load All Data ------------------------------------------------------
all_set = load_data_from_folder(ALL_DATA_FOLDER)
all_set = np.array(all_set, dtype=object)
print(f"📦 Total samples found: {len(all_set)}")

# --------------------------------------------⚡️ Objective Function for Optuna (F1-Score) --------------------------------------------
def objective(trial):
    # 1. Suggest hyperparameters for this trial
    patch_size = trial.suggest_categorical('patch_size', [128, 160, 256, 320])
    projection_dim = trial.suggest_categorical('projection_dim', [64, 128])
    num_heads = trial.suggest_categorical('num_heads', [4, 8])
    transformer_layers = trial.suggest_int('transformer_layers', 2, 6)
    ff_dim = trial.suggest_categorical('ff_dim', [128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    
    # If hyperparameters are invalid, prune the trial
    if projection_dim % num_heads != 0:
        raise optuna.exceptions.TrialPruned()
        
    # ✅ Ensure patch_size is compatible with SEQ_LEN
    if SEQ_LEN % patch_size != 0:
        print(f"Pruning trial: SEQ_LEN ({SEQ_LEN}) not divisible by patch_size ({patch_size})")
        raise optuna.exceptions.TrialPruned()


    print(f"\n🚀 Starting Trial {trial.number} with params: Patch={patch_size}, Proj_dim={projection_dim}, Heads={num_heads}, Layers={transformer_layers}, LR={lr:.5f}, Batch={batch_size}")
    
    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(KF.split(all_set)):
        print(f"--- Fold {fold + 1}/{N_SPLITS} ---")
        train_data = all_set[train_idx]
        val_data = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM)
        val_gen = EEGDataGenerator(val_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM, shuffle=False)
        
        model = build_eeg_transformer_vit_style(
            SEQ_LEN, INPUT_DIM, patch_size, projection_dim, num_heads,
            transformer_layers, ff_dim, dropout_rate, lr
        )

        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # ✅ Calculate F1-Score for evaluation
        y_pred_probs = model.predict(val_gen, verbose=0)
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()
        
        y_true = np.array([label for _, label in val_data])
        
        # Ensure vectors have the same length
        min_len = min(len(y_true), len(y_pred))
        
        if min_len == 0:
            print("Warning: Validation generator returned 0 samples. Skipping fold.")
            continue
            
        current_f1 = f1_score(y_true[:min_len], y_pred[:min_len])
        fold_f1_scores.append(current_f1)
        
    if not fold_f1_scores:
        print(f"Trial {trial.number} failed, no F1 scores recorded.")
        return 0.0 # Or another penalty value

    # ✅ Return the mean F1-Score for optimization
    mean_f1 = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1:.4f}")
    
    return mean_f1

# ------------------------------------------------------⚡️ Run the Optuna Study ------------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# -------------------------------------------------⚡️ Print the results of the study -------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial
print("\n🎉 Best trial:")
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}") # ✅ Display the best F1-Score

print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-11-01 12:49:23,075] A new study created in memory with name: no-name-95fa6d48-ff2c-4e1b-be0b-e0374056ede4


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: Patch=160, Proj_dim=64, Heads=8, Layers=3, LR=0.00051, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-01 13:49:14,269] Trial 0 finished with value: 0.6053743518465453 and parameters: {'patch_size': 160, 'projection_dim': 64, 'num_heads': 8, 'transformer_layers': 3, 'ff_dim': 128, 'dropout_rate': 0.14243113891410897, 'learning_rate': 0.0005136754630573811, 'batch_size': 16}. Best is trial 0 with value: 0.6053743518465453.


✅ Trial 0 finished with Mean F1-Score: 0.6054

🚀 Starting Trial 1 with params: Patch=128, Proj_dim=128, Heads=8, Layers=6, LR=0.00092, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-01 16:20:12,165] Trial 1 finished with value: 0.6034065321048924 and parameters: {'patch_size': 128, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 6, 'ff_dim': 128, 'dropout_rate': 0.3005847620841875, 'learning_rate': 0.0009192834941783439, 'batch_size': 16}. Best is trial 0 with value: 0.6053743518465453.


...

✅ Trial 7 finished with Mean F1-Score: 0.6007

🚀 Starting Trial 8 with params: Patch=320, Proj_dim=64, Heads=8, Layers=4, LR=0.00093, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-01 21:40:00,639] Trial 8 finished with value: 0.7023106929621501 and parameters: {'patch_size': 320, 'projection_dim': 64, 'num_heads': 8, 'transformer_layers': 4, 'ff_dim': 128, 'dropout_rate': 0.12974496231338048, 'learning_rate': 0.0009273877043023403, 'batch_size': 32}. Best is trial 2 with value: 0.7283726486178488.


✅ Trial 8 finished with Mean F1-Score: 0.7023

🚀 Starting Trial 9 with params: Patch=320, Proj_dim=128, Heads=8, Layers=3, LR=0.00019, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-01 22:16:46,992] Trial 9 finished with value: 0.7321580479819094 and parameters: {'patch_size': 320, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 3, 'ff_dim': 256, 'dropout_rate': 0.3755571227807356, 'learning_rate': 0.00018907008232065594, 'batch_size': 32}. Best is trial 9 with value: 0.7321580479819094.


✅ Trial 9 finished with Mean F1-Score: 0.7322

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.7322

  Best Hyperparameters:
    patch_size: 320
    projection_dim: 128
    num_heads: 8
    transformer_layers: 3
    ff_dim: 256
    dropout_rate: 0.3755571227807356
    learning_rate: 0.00018907008232065594
    batch_size: 32


In [3]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
                                     Dropout, GlobalAveragePooling1D, Embedding)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
# Fixed settings for optimization
SEQ_LEN = 2560
INPUT_DIM = 27
N_SPLITS = 5
KF = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
ALL_DATA_FOLDER = "Raw/All Data"
EPOCHS = 70

# --------------------------------------------------------⚡️ Custom Model Layers ---------------------------------------------------------
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

class PatchEncoder(Layer):
    """
    Encodes each EEG patch into a vector and adds a fixed Sinusoidal position embedding.
    """
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = Dense(units=projection_dim)
        
        # --- ✅ Create the fixed Sinusoidal Positional Embedding ---
        pos_encoding_matrix = self.get_sinusoidal_position_embedding(
            num_patches, projection_dim
        )
        self.position_embedding = tf.Variable(
            pos_encoding_matrix,
            trainable=False, # This parameter is not learned
            name="position_embedding"
        )

    def get_sinusoidal_position_embedding(self, sequence_length, embedding_dim):
        """Generates the sinusoidal positional embedding matrix."""
        if embedding_dim % 2 != 0:
            raise ValueError(
                f"Cannot create sinusoidal embedding for odd dimension {embedding_dim}. "
                "Use an even dimension."
            )
            
        angle_rates = 1.0 / (10000 ** (
            tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)
        ))
        positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
        angle_rads = positions * angle_rates[tf.newaxis, :]
        sines = tf.math.sin(angle_rads)
        cosines = tf.math.cos(angle_rads)
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        
        return pos_encoding # shape [sequence_length, embedding_dim]

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        patch_height = patch.shape[-2]
        patch_width = patch.shape[-1]
        
        patch_flattened = tf.reshape(patch, [batch_size, self.num_patches, patch_height * patch_width])
        encoded = self.projection(patch_flattened)
        
        # --- ✅ Add the fixed embedding matrix ---
        encoded += self.position_embedding 
        
        return encoded

    def get_config(self):
        config = super(PatchEncoder, self).get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config

# -----------------------------------⚡️ Function to build the ViT model (Accepts Hyperparameters) ------------------------------------
# ✅ This function now automatically uses the new PatchEncoder
def build_eeg_transformer_vit_style(seq_len, input_dim, patch_size, projection_dim, num_heads,
                                     transformer_layers, ff_dim, dropout_rate, lr):
    # ✅ Important assertions to prevent architectural errors
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    assert projection_dim % num_heads == 0, "Projection dim must be divisible by number of heads."

    num_patches = seq_len // patch_size
    inputs = Input(shape=(seq_len, input_dim))
    
    patches = Patches(patch_size)(inputs)
    # ✅ This line now uses the Sinusoidal PatchEncoder class defined above
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    outputs = Dense(1, activation='sigmoid')(outputs)
    
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------------------------------⚡️ Helper Functions -------------------------------------------------------
def load_data_from_folder(folder):
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, seq_len=2560, input_dim=27, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        data = np.zeros((len(batch_data_list), self.seq_len, self.input_dim))
        labels = np.zeros((len(batch_data_list),))
        
        for i, (file, label) in enumerate(batch_data_list):
            sample = np.load(file)
            # This model requires the shape (signal_length, num_channels)
            # ✅ Ensure correct data shape based on your code
            if sample.shape == (self.input_dim, self.seq_len):
                sample = sample.T
            
            if sample.shape == (self.seq_len, self.input_dim):
                data[i,] = sample
                labels[i] = label
            else:
                # If data doesn't have the expected shape, log it
                print(f"Warning: Skipping file {file} with unexpected shape {sample.shape}")

        # Filtering samples that might not have been loaded (though this won't happen with np.zeros)
        return data, labels

# ------------------------------------------------------⚡️ Load All Data ------------------------------------------------------
all_set = load_data_from_folder(ALL_DATA_FOLDER)
all_set = np.array(all_set, dtype=object)
print(f"📦 Total samples found: {len(all_set)}")

# --------------------------------------------⚡️ Objective Function for Optuna (F1-Score) --------------------------------------------
def objective(trial):
    # 1. Suggest hyperparameters for this trial
    patch_size = trial.suggest_categorical('patch_size', [128, 160, 256, 320])
    projection_dim = trial.suggest_categorical('projection_dim', [64, 128])
    num_heads = trial.suggest_categorical('num_heads', [4, 8])
    transformer_layers = trial.suggest_int('transformer_layers', 2, 6)
    ff_dim = trial.suggest_categorical('ff_dim', [128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    
    # If hyperparameters are invalid, prune the trial
    if projection_dim % num_heads != 0:
        raise optuna.exceptions.TrialPruned()
        
    # ✅ Ensure patch_size is compatible with SEQ_LEN
    if SEQ_LEN % patch_size != 0:
        print(f"Pruning trial: SEQ_LEN ({SEQ_LEN}) not divisible by patch_size ({patch_size})")
        raise optuna.exceptions.TrialPruned()


    print(f"\n🚀 Starting Trial {trial.number} with params: Patch={patch_size}, Proj_dim={projection_dim}, Heads={num_heads}, Layers={transformer_layers}, LR={lr:.5f}, Batch={batch_size}")
    
    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(KF.split(all_set)):
        print(f"--- Fold {fold + 1}/{N_SPLITS} ---")
        train_data = all_set[train_idx]
        val_data = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM)
        val_gen = EEGDataGenerator(val_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM, shuffle=False)
        
        model = build_eeg_transformer_vit_style(
            SEQ_LEN, INPUT_DIM, patch_size, projection_dim, num_heads,
            transformer_layers, ff_dim, dropout_rate, lr
        )

        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # ✅ Calculate F1-Score for evaluation
        y_pred_probs = model.predict(val_gen, verbose=0)
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()
        
        y_true = np.array([label for _, label in val_data])
        
        # Ensure vectors have the same length
        min_len = min(len(y_true), len(y_pred))
        
        if min_len == 0:
            print("Warning: Validation generator returned 0 samples. Skipping fold.")
            continue
            
        current_f1 = f1_score(y_true[:min_len], y_pred[:min_len])
        fold_f1_scores.append(current_f1)
        
    if not fold_f1_scores:
        print(f"Trial {trial.number} failed, no F1 scores recorded.")
        return 0.0 # Or another penalty value

    # ✅ Return the mean F1-Score for optimization
    mean_f1 = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1:.4f}")
    
    return mean_f1

# ------------------------------------------------------⚡️ Run the Optuna Study ------------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# -------------------------------------------------⚡️ Print the results of the study -------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial
print("\n🎉 Best trial:")
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}") # ✅ Display the best F1-Score

print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-11-01 22:16:47,913] A new study created in memory with name: no-name-d6ccfba5-08be-4c14-b658-cfc67360b49f


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: Patch=256, Proj_dim=64, Heads=4, Layers=5, LR=0.00029, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-01 23:14:41,897] Trial 0 finished with value: 0.6482730069772628 and parameters: {'patch_size': 256, 'projection_dim': 64, 'num_heads': 4, 'transformer_layers': 5, 'ff_dim': 128, 'dropout_rate': 0.35029939105986174, 'learning_rate': 0.00029076690841317734, 'batch_size': 16}. Best is trial 0 with value: 0.6482730069772628.


✅ Trial 0 finished with Mean F1-Score: 0.6483

🚀 Starting Trial 1 with params: Patch=160, Proj_dim=128, Heads=8, Layers=3, LR=0.00039, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-01 23:43:19,157] Trial 1 finished with value: 0.6654267831584619 and parameters: {'patch_size': 160, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 3, 'ff_dim': 128, 'dropout_rate': 0.2655045708876721, 'learning_rate': 0.0003948465960282758, 'batch_size': 32}. Best is trial 1 with value: 0.6654267831584619.


...

✅ Trial 7 finished with Mean F1-Score: 0.6478

🚀 Starting Trial 8 with params: Patch=320, Proj_dim=128, Heads=8, Layers=2, LR=0.00016, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-02 04:03:29,666] Trial 8 finished with value: 0.668791937651004 and parameters: {'patch_size': 320, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 2, 'ff_dim': 128, 'dropout_rate': 0.3820646178430742, 'learning_rate': 0.00016098757986394944, 'batch_size': 32}. Best is trial 6 with value: 0.6841118419129286.


✅ Trial 8 finished with Mean F1-Score: 0.6688

🚀 Starting Trial 9 with params: Patch=128, Proj_dim=64, Heads=8, Layers=3, LR=0.00012, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-02 04:56:11,710] Trial 9 finished with value: 0.6689679339321547 and parameters: {'patch_size': 128, 'projection_dim': 64, 'num_heads': 8, 'transformer_layers': 3, 'ff_dim': 256, 'dropout_rate': 0.2805958900927098, 'learning_rate': 0.00011639955505462312, 'batch_size': 16}. Best is trial 6 with value: 0.6841118419129286.


✅ Trial 9 finished with Mean F1-Score: 0.6690

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.6841

  Best Hyperparameters:
    patch_size: 320
    projection_dim: 128
    num_heads: 4
    transformer_layers: 2
    ff_dim: 256
    dropout_rate: 0.3409713547639843
    learning_rate: 0.00019259454444870477
    batch_size: 32


In [6]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
                                     Dropout, GlobalAveragePooling1D, Embedding)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
# Fixed settings for optimization
SEQ_LEN = 2560
INPUT_DIM = 27
N_SPLITS = 5
KF = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
ALL_DATA_FOLDER = "Filtered_8_30Hz"
EPOCHS = 70

# --------------------------------------------------------⚡️ Custom Model Layers ---------------------------------------------------------
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

class PatchEncoder(Layer):
    """
    Encodes each EEG patch into a vector and adds a fixed Sinusoidal position embedding.
    """
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = Dense(units=projection_dim)
        
        # --- ✅ Create the fixed Sinusoidal Positional Embedding ---
        pos_encoding_matrix = self.get_sinusoidal_position_embedding(
            num_patches, projection_dim
        )
        self.position_embedding = tf.Variable(
            pos_encoding_matrix,
            trainable=False, # This parameter is not learned
            name="position_embedding"
        )

    def get_sinusoidal_position_embedding(self, sequence_length, embedding_dim):
        """Generates the sinusoidal positional embedding matrix."""
        if embedding_dim % 2 != 0:
            raise ValueError(
                f"Cannot create sinusoidal embedding for odd dimension {embedding_dim}. "
                "Use an even dimension."
            )
            
        angle_rates = 1.0 / (10000 ** (
            tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)
        ))
        positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
        angle_rads = positions * angle_rates[tf.newaxis, :]
        sines = tf.math.sin(angle_rads)
        cosines = tf.math.cos(angle_rads)
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        
        return pos_encoding # shape [sequence_length, embedding_dim]

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        patch_height = patch.shape[-2]
        patch_width = patch.shape[-1]
        
        patch_flattened = tf.reshape(patch, [batch_size, self.num_patches, patch_height * patch_width])
        encoded = self.projection(patch_flattened)
        
        # --- ✅ Add the fixed embedding matrix ---
        encoded += self.position_embedding 
        
        return encoded

    def get_config(self):
        config = super(PatchEncoder, self).get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config

# -----------------------------------⚡️ Function to build the ViT model (Accepts Hyperparameters) ------------------------------------
# ✅ This function now automatically uses the new PatchEncoder
def build_eeg_transformer_vit_style(seq_len, input_dim, patch_size, projection_dim, num_heads,
                                     transformer_layers, ff_dim, dropout_rate, lr):
    # ✅ Important assertions to prevent architectural errors
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    assert projection_dim % num_heads == 0, "Projection dim must be divisible by number of heads."

    num_patches = seq_len // patch_size
    inputs = Input(shape=(seq_len, input_dim))
    
    patches = Patches(patch_size)(inputs)
    # ✅ This line now uses the Sinusoidal PatchEncoder class defined above
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    outputs = Dense(1, activation='sigmoid')(outputs)
    
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------------------------------⚡️ Helper Functions -------------------------------------------------------
def load_data_from_folder(folder):
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, seq_len=2560, input_dim=27, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        data = np.zeros((len(batch_data_list), self.seq_len, self.input_dim))
        labels = np.zeros((len(batch_data_list),))
        
        for i, (file, label) in enumerate(batch_data_list):
            sample = np.load(file)
            # This model requires the shape (signal_length, num_channels)
            # ✅ Ensure correct data shape based on your code
            if sample.shape == (self.input_dim, self.seq_len):
                sample = sample.T
            
            if sample.shape == (self.seq_len, self.input_dim):
                data[i,] = sample
                labels[i] = label
            else:
                # If data doesn't have the expected shape, log it
                print(f"Warning: Skipping file {file} with unexpected shape {sample.shape}")

        # Filtering samples that might not have been loaded (though this won't happen with np.zeros)
        return data, labels

# ------------------------------------------------------⚡️ Load All Data ------------------------------------------------------
all_set = load_data_from_folder(ALL_DATA_FOLDER)
all_set = np.array(all_set, dtype=object)
print(f"📦 Total samples found: {len(all_set)}")

# --------------------------------------------⚡️ Objective Function for Optuna (F1-Score) --------------------------------------------
def objective(trial):
    # 1. Suggest hyperparameters for this trial
    patch_size = trial.suggest_categorical('patch_size', [128, 160, 256, 320])
    projection_dim = trial.suggest_categorical('projection_dim', [64, 128])
    num_heads = trial.suggest_categorical('num_heads', [4, 8])
    transformer_layers = trial.suggest_int('transformer_layers', 2, 6)
    ff_dim = trial.suggest_categorical('ff_dim', [128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    
    # If hyperparameters are invalid, prune the trial
    if projection_dim % num_heads != 0:
        raise optuna.exceptions.TrialPruned()
        
    # ✅ Ensure patch_size is compatible with SEQ_LEN
    if SEQ_LEN % patch_size != 0:
        print(f"Pruning trial: SEQ_LEN ({SEQ_LEN}) not divisible by patch_size ({patch_size})")
        raise optuna.exceptions.TrialPruned()


    print(f"\n🚀 Starting Trial {trial.number} with params: Patch={patch_size}, Proj_dim={projection_dim}, Heads={num_heads}, Layers={transformer_layers}, LR={lr:.5f}, Batch={batch_size}")
    
    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(KF.split(all_set)):
        print(f"--- Fold {fold + 1}/{N_SPLITS} ---")
        train_data = all_set[train_idx]
        val_data = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM)
        val_gen = EEGDataGenerator(val_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM, shuffle=False)
        
        model = build_eeg_transformer_vit_style(
            SEQ_LEN, INPUT_DIM, patch_size, projection_dim, num_heads,
            transformer_layers, ff_dim, dropout_rate, lr
        )

        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # ✅ Calculate F1-Score for evaluation
        y_pred_probs = model.predict(val_gen, verbose=0)
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()
        
        y_true = np.array([label for _, label in val_data])
        
        # Ensure vectors have the same length
        min_len = min(len(y_true), len(y_pred))
        
        if min_len == 0:
            print("Warning: Validation generator returned 0 samples. Skipping fold.")
            continue
            
        current_f1 = f1_score(y_true[:min_len], y_pred[:min_len])
        fold_f1_scores.append(current_f1)
        
    if not fold_f1_scores:
        print(f"Trial {trial.number} failed, no F1 scores recorded.")
        return 0.0 # Or another penalty value

    # ✅ Return the mean F1-Score for optimization
    mean_f1 = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1:.4f}")
    
    return mean_f1

# ------------------------------------------------------⚡️ Run the Optuna Study ------------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# -------------------------------------------------⚡️ Print the results of the study -------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial
print("\n🎉 Best trial:")
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}") # ✅ Display the best F1-Score

print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-11-02 21:42:46,620] A new study created in memory with name: no-name-306d74b5-ca74-40b3-ad0e-ecd84d2169c7


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: Patch=128, Proj_dim=64, Heads=4, Layers=6, LR=0.00033, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-02 23:03:12,034] Trial 0 finished with value: 0.6917031454959159 and parameters: {'patch_size': 128, 'projection_dim': 64, 'num_heads': 4, 'transformer_layers': 6, 'ff_dim': 256, 'dropout_rate': 0.20741246955089163, 'learning_rate': 0.00033106533228438855, 'batch_size': 16}. Best is trial 0 with value: 0.6917031454959159.


✅ Trial 0 finished with Mean F1-Score: 0.6917

🚀 Starting Trial 1 with params: Patch=128, Proj_dim=128, Heads=8, Layers=4, LR=0.00071, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-02 23:51:30,639] Trial 1 finished with value: 0.6581984936874765 and parameters: {'patch_size': 128, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 4, 'ff_dim': 128, 'dropout_rate': 0.34752531654361735, 'learning_rate': 0.000710999916072638, 'batch_size': 32}. Best is trial 0 with value: 0.6917031454959159.


...

✅ Trial 7 finished with Mean F1-Score: 0.7045

🚀 Starting Trial 8 with params: Patch=128, Proj_dim=128, Heads=8, Layers=2, LR=0.00017, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-03 07:05:38,727] Trial 8 finished with value: 0.7660007268932439 and parameters: {'patch_size': 128, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 2, 'ff_dim': 128, 'dropout_rate': 0.36094771336885223, 'learning_rate': 0.0001730522300521005, 'batch_size': 32}. Best is trial 8 with value: 0.7660007268932439.


✅ Trial 8 finished with Mean F1-Score: 0.7660

🚀 Starting Trial 9 with params: Patch=160, Proj_dim=64, Heads=8, Layers=2, LR=0.00021, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-03 08:06:32,121] Trial 9 finished with value: 0.7261700307189435 and parameters: {'patch_size': 160, 'projection_dim': 64, 'num_heads': 8, 'transformer_layers': 2, 'ff_dim': 128, 'dropout_rate': 0.16228978788302223, 'learning_rate': 0.0002092465336213602, 'batch_size': 32}. Best is trial 8 with value: 0.7660007268932439.


✅ Trial 9 finished with Mean F1-Score: 0.7262

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.7660

  Best Hyperparameters:
    patch_size: 128
    projection_dim: 128
    num_heads: 8
    transformer_layers: 2
    ff_dim: 128
    dropout_rate: 0.36094771336885223
    learning_rate: 0.0001730522300521005
    batch_size: 32


In [1]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# --------------------------------------------------------⚡️ Import Necessary libraries -------------------------------------------------------
import os
import numpy as np
import tensorflow as tf
import optuna
from tensorflow.keras.layers import (Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
                                     Dropout, GlobalAveragePooling1D, Embedding)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------⚡️ Initial settings -------------------------------------------------------------
# Fixed settings for optimization
SEQ_LEN = 2560
INPUT_DIM = 27
N_SPLITS = 5
KF = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
ALL_DATA_FOLDER = "Filtered_8_13Hz"
EPOCHS = 70

# --------------------------------------------------------⚡️ Custom Model Layers ---------------------------------------------------------
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

class PatchEncoder(Layer):
    """
    Encodes each EEG patch into a vector and adds a fixed Sinusoidal position embedding.
    """
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = Dense(units=projection_dim)
        
        # --- ✅ Create the fixed Sinusoidal Positional Embedding ---
        pos_encoding_matrix = self.get_sinusoidal_position_embedding(
            num_patches, projection_dim
        )
        self.position_embedding = tf.Variable(
            pos_encoding_matrix,
            trainable=False, # This parameter is not learned
            name="position_embedding"
        )

    def get_sinusoidal_position_embedding(self, sequence_length, embedding_dim):
        """Generates the sinusoidal positional embedding matrix."""
        if embedding_dim % 2 != 0:
            raise ValueError(
                f"Cannot create sinusoidal embedding for odd dimension {embedding_dim}. "
                "Use an even dimension."
            )
            
        angle_rates = 1.0 / (10000 ** (
            tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)
        ))
        positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
        angle_rads = positions * angle_rates[tf.newaxis, :]
        sines = tf.math.sin(angle_rads)
        cosines = tf.math.cos(angle_rads)
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        
        return pos_encoding # shape [sequence_length, embedding_dim]

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        patch_height = patch.shape[-2]
        patch_width = patch.shape[-1]
        
        patch_flattened = tf.reshape(patch, [batch_size, self.num_patches, patch_height * patch_width])
        encoded = self.projection(patch_flattened)
        
        # --- ✅ Add the fixed embedding matrix ---
        encoded += self.position_embedding 
        
        return encoded

    def get_config(self):
        config = super(PatchEncoder, self).get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config

# -----------------------------------⚡️ Function to build the ViT model (Accepts Hyperparameters) ------------------------------------
# ✅ This function now automatically uses the new PatchEncoder
def build_eeg_transformer_vit_style(seq_len, input_dim, patch_size, projection_dim, num_heads,
                                     transformer_layers, ff_dim, dropout_rate, lr):
    # ✅ Important assertions to prevent architectural errors
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    assert projection_dim % num_heads == 0, "Projection dim must be divisible by number of heads."

    num_patches = seq_len // patch_size
    inputs = Input(shape=(seq_len, input_dim))
    
    patches = Patches(patch_size)(inputs)
    # ✅ This line now uses the Sinusoidal PatchEncoder class defined above
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    outputs = Dense(1, activation='sigmoid')(outputs)
    
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------------------------------⚡️ Helper Functions -------------------------------------------------------
def load_data_from_folder(folder):
    dataset = []
    for file in os.listdir(folder):
        if file.endswith(".npy"):
            label = 1 if "right" in file.lower() else 0
            path = os.path.join(folder, file)
            dataset.append((path, label))
    return dataset

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size=32, seq_len=2560, input_dim=27, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_data_list = [self.dataset[k] for k in batch_indexes]
        X, y = self.__data_generation(batch_data_list)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_data_list):
        data = np.zeros((len(batch_data_list), self.seq_len, self.input_dim))
        labels = np.zeros((len(batch_data_list),))
        
        for i, (file, label) in enumerate(batch_data_list):
            sample = np.load(file)
            # This model requires the shape (signal_length, num_channels)
            # ✅ Ensure correct data shape based on your code
            if sample.shape == (self.input_dim, self.seq_len):
                sample = sample.T
            
            if sample.shape == (self.seq_len, self.input_dim):
                data[i,] = sample
                labels[i] = label
            else:
                # If data doesn't have the expected shape, log it
                print(f"Warning: Skipping file {file} with unexpected shape {sample.shape}")

        # Filtering samples that might not have been loaded (though this won't happen with np.zeros)
        return data, labels

# ------------------------------------------------------⚡️ Load All Data ------------------------------------------------------
all_set = load_data_from_folder(ALL_DATA_FOLDER)
all_set = np.array(all_set, dtype=object)
print(f"📦 Total samples found: {len(all_set)}")

# --------------------------------------------⚡️ Objective Function for Optuna (F1-Score) --------------------------------------------
def objective(trial):
    # 1. Suggest hyperparameters for this trial
    patch_size = trial.suggest_categorical('patch_size', [128, 160, 256, 320])
    projection_dim = trial.suggest_categorical('projection_dim', [64, 128])
    num_heads = trial.suggest_categorical('num_heads', [4, 8])
    transformer_layers = trial.suggest_int('transformer_layers', 2, 6)
    ff_dim = trial.suggest_categorical('ff_dim', [128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    
    # If hyperparameters are invalid, prune the trial
    if projection_dim % num_heads != 0:
        raise optuna.exceptions.TrialPruned()
        
    # ✅ Ensure patch_size is compatible with SEQ_LEN
    if SEQ_LEN % patch_size != 0:
        print(f"Pruning trial: SEQ_LEN ({SEQ_LEN}) not divisible by patch_size ({patch_size})")
        raise optuna.exceptions.TrialPruned()


    print(f"\n🚀 Starting Trial {trial.number} with params: Patch={patch_size}, Proj_dim={projection_dim}, Heads={num_heads}, Layers={transformer_layers}, LR={lr:.5f}, Batch={batch_size}")
    
    fold_f1_scores = []
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    for fold, (train_idx, val_idx) in enumerate(KF.split(all_set)):
        print(f"--- Fold {fold + 1}/{N_SPLITS} ---")
        train_data = all_set[train_idx]
        val_data = all_set[val_idx]
        
        train_gen = EEGDataGenerator(train_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM)
        val_gen = EEGDataGenerator(val_data, batch_size, seq_len=SEQ_LEN, input_dim=INPUT_DIM, shuffle=False)
        
        model = build_eeg_transformer_vit_style(
            SEQ_LEN, INPUT_DIM, patch_size, projection_dim, num_heads,
            transformer_layers, ff_dim, dropout_rate, lr
        )

        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # ✅ Calculate F1-Score for evaluation
        y_pred_probs = model.predict(val_gen, verbose=0)
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()
        
        y_true = np.array([label for _, label in val_data])
        
        # Ensure vectors have the same length
        min_len = min(len(y_true), len(y_pred))
        
        if min_len == 0:
            print("Warning: Validation generator returned 0 samples. Skipping fold.")
            continue
            
        current_f1 = f1_score(y_true[:min_len], y_pred[:min_len])
        fold_f1_scores.append(current_f1)
        
    if not fold_f1_scores:
        print(f"Trial {trial.number} failed, no F1 scores recorded.")
        return 0.0 # Or another penalty value

    # ✅ Return the mean F1-Score for optimization
    mean_f1 = np.mean(fold_f1_scores)
    print(f"✅ Trial {trial.number} finished with Mean F1-Score: {mean_f1:.4f}")
    
    return mean_f1

# ------------------------------------------------------⚡️ Run the Optuna Study ------------------------------------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# -------------------------------------------------⚡️ Print the results of the study -------------------------------------------------
print("\n" + "="*50)
print("🏆 Optuna Hyperparameter Optimization Finished!")
print("="*50)
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial
print("\n🎉 Best trial:")
print(f"  Value (Mean F1-Score): {best_trial.value:.4f}") # ✅ Display the best F1-Score

print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-11-15 16:29:29,078] A new study created in memory with name: no-name-7e441dbb-9615-4bef-9d3a-7f341112c119


📦 Total samples found: 20792

🚀 Starting Trial 0 with params: Patch=256, Proj_dim=64, Heads=8, Layers=6, LR=0.00066, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-15 17:49:04,631] Trial 0 finished with value: 0.6865139997358776 and parameters: {'patch_size': 256, 'projection_dim': 64, 'num_heads': 8, 'transformer_layers': 6, 'ff_dim': 256, 'dropout_rate': 0.2348467322207308, 'learning_rate': 0.0006612605334253023, 'batch_size': 32}. Best is trial 0 with value: 0.6865139997358776.


✅ Trial 0 finished with Mean F1-Score: 0.6865

🚀 Starting Trial 1 with params: Patch=256, Proj_dim=128, Heads=4, Layers=5, LR=0.00020, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-15 19:10:40,568] Trial 1 finished with value: 0.6985166016994903 and parameters: {'patch_size': 256, 'projection_dim': 128, 'num_heads': 4, 'transformer_layers': 5, 'ff_dim': 128, 'dropout_rate': 0.3686856286561532, 'learning_rate': 0.00020141182516951402, 'batch_size': 16}. Best is trial 1 with value: 0.6985166016994903.


...

✅ Trial 7 finished with Mean F1-Score: 0.6920

🚀 Starting Trial 8 with params: Patch=160, Proj_dim=64, Heads=8, Layers=4, LR=0.00094, Batch=32
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-16 03:32:21,872] Trial 8 finished with value: 0.6954998822065913 and parameters: {'patch_size': 160, 'projection_dim': 64, 'num_heads': 8, 'transformer_layers': 4, 'ff_dim': 256, 'dropout_rate': 0.15208956568526638, 'learning_rate': 0.0009393119923039384, 'batch_size': 32}. Best is trial 3 with value: 0.7248060964898368.


✅ Trial 8 finished with Mean F1-Score: 0.6955

🚀 Starting Trial 9 with params: Patch=128, Proj_dim=64, Heads=4, Layers=6, LR=0.00019, Batch=16
--- Fold 1/5 ---
--- Fold 2/5 ---
--- Fold 3/5 ---
--- Fold 4/5 ---
--- Fold 5/5 ---


[I 2025-11-16 05:37:40,106] Trial 9 finished with value: 0.7655905906849465 and parameters: {'patch_size': 128, 'projection_dim': 64, 'num_heads': 4, 'transformer_layers': 6, 'ff_dim': 128, 'dropout_rate': 0.376254387977631, 'learning_rate': 0.0001857994136619948, 'batch_size': 16}. Best is trial 9 with value: 0.7655905906849465.


✅ Trial 9 finished with Mean F1-Score: 0.7656

🏆 Optuna Hyperparameter Optimization Finished!
Number of finished trials: 10

🎉 Best trial:
  Value (Mean F1-Score): 0.7656

  Best Hyperparameters:
    patch_size: 128
    projection_dim: 64
    num_heads: 4
    transformer_layers: 6
    ff_dim: 128
    dropout_rate: 0.376254387977631
    learning_rate: 0.0001857994136619948
    batch_size: 16
